# 02-12b Kaggle nnU-Net v2 PHE-only baseline with CT window/skull-strip preprocessing

Notebook này là bản Kaggle của `02_12b`, vẫn giữ vai trò baseline PHE-only:

- Chỉ dùng `PHE-SICH-CT-IDS` vì đây là bộ có manual PHE mask.
- Không dùng `Seg-CQ500`, `Instance2022`, ICH teacher, pseudo-ICH prior, hoặc PHE-derived pseudo ICH.
- Target vẫn là binary segmentation: `background` và `PHE`.
- Giữ cùng split `99/10/11` như `02_12` để so sánh công bằng.
- Khác biệt chính so với `02_12`: CT input = brain window L=40/W=120 + subdural window L=75/W=215, đều skull-stripped bằng mask nội sọ tạo từ CT.

Kaggle path policy:

```text
input  : auto-detect under /kaggle/input
output : /kaggle/working/outputs_02_12b_kaggle/bwss
result : /kaggle/working/o12bk_results/bwss
```

Lưu ý: notebook này đã bật các flag chạy thật. Khi Run All, nó sẽ convert data -> plan/preprocess -> ghi split -> train -> predict -> evaluate.


In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import time

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input') if IS_KAGGLE else None
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()
PROJECT_ROOT = WORK_ROOT

# Optional manual overrides. Leave None for auto-detect.
# Example Kaggle values:
# USER_NNUNET_REPO = Path('/kaggle/input/<your-nnunet-dataset>/nnUNet-master')
# USER_PHE_ROOT = Path('/kaggle/input/<your-phe-dataset>/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
# USER_SPLIT_CSV = Path('/kaggle/input/<your-split-dataset>/3dff_iph_phe_patient_split.csv')
USER_NNUNET_REPO = None
USER_PHE_ROOT = None
USER_SPLIT_CSV = None


def has_nifti_pair_root(path_like):
    root = Path(path_like)
    if not (root / 'set').exists() or not (root / 'label').exists():
        return False
    image_count = len(list((root / 'set').glob('*.nii'))) + len(list((root / 'set').glob('*.nii.gz')))
    mask_count = len(list((root / 'label').glob('*.nii'))) + len(list((root / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_nnunet_repo():
    if USER_NNUNET_REPO is not None:
        repo = Path(USER_NNUNET_REPO)
        if (repo / 'nnunetv2' / '__init__.py').exists():
            return repo
        raise FileNotFoundError(f'USER_NNUNET_REPO is not an nnU-Net repo: {repo}')
    candidates = [LOCAL_ROOT / 'nnUNet-master', LOCAL_ROOT]
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.iterdir())
    for base in candidates:
        if (base / 'nnunetv2' / '__init__.py').exists():
            return base
        if (base / 'nnUNet-master' / 'nnunetv2' / '__init__.py').exists():
            return base / 'nnUNet-master'
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    for root in search_roots:
        for init_file in root.rglob('nnunetv2/__init__.py'):
            return init_file.parents[1]
    print('WARNING: Could not find nnU-Net repo under /kaggle/input. The dependency cell will try pip install/import nnunetv2 instead.')
    return None


def find_phe_root():
    if USER_PHE_ROOT is not None:
        root = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(root):
            return root
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {root}')
    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for root in candidates:
        if has_nifti_pair_root(root):
            return root
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            candidate = set_dir.parent
            if has_nifti_pair_root(candidate):
                found.append(candidate)
    if found:
        found = sorted(found, key=lambda x: (('SubdatasetA_NIFIT' not in str(x)), len(str(x))))
        return found[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with set/ and label/. Set USER_PHE_ROOT manually.')


def find_split_csv():
    if USER_SPLIT_CSV is not None and Path(USER_SPLIT_CSV).exists():
        return Path(USER_SPLIT_CSV)
    names = {'3dff_iph_phe_patient_split.csv', 'phe_split.csv', 'patient_split.csv'}
    candidates = [
        LOCAL_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'
    ]
    if IS_KAGGLE:
        for root in KAGGLE_INPUT.iterdir():
            for csv_path in root.rglob('*.csv'):
                if csv_path.name in names:
                    candidates.append(csv_path)
    for csv_path in candidates:
        if csv_path.exists():
            return csv_path
    return None


NNUNET_REPO = find_nnunet_repo()
PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = find_split_csv()

# Short paths avoid Windows/Kaggle path clutter during nnU-Net checkpointing/evaluation.
OUTPUT_ROOT = WORK_ROOT / 'outputs_02_12b_kaggle' / 'bwss'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = WORK_ROOT / 'o12bk_results' / 'bwss'

DATASET_ID = 122
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_12b_bwss'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

# Same CT preprocessing idea as 19b, but without any pseudo-ICH/prior channels.
CT_PREPROCESS_PROFILE = 'brain_subdural_window_skullstrip_v1'
CT_WINDOW_OUTPUT_SCALE = 100.0
CT_WINDOW_LIBRARY = {
    'brain': {'level': 40.0, 'width': 120.0},
    'subdural': {'level': 75.0, 'width': 215.0},
    'bone': {'level': 600.0, 'width': 2800.0},
}
CT_INPUT_WINDOW_NAMES = ['brain', 'subdural']
missing_ct_windows = [name for name in CT_INPUT_WINDOW_NAMES if name not in CT_WINDOW_LIBRARY]
assert not missing_ct_windows, f'Unknown CT input windows: {missing_ct_windows}'
CT_INPUT_WINDOWS = []
for window_name in CT_INPUT_WINDOW_NAMES:
    spec = dict(CT_WINDOW_LIBRARY[window_name])
    spec['name'] = window_name
    spec['low'] = float(spec['level']) - float(spec['width']) / 2.0
    spec['high'] = float(spec['level']) + float(spec['width']) / 2.0
    CT_INPUT_WINDOWS.append(spec)

CT_SKULL_STRIP_ENABLED = True
CT_SKULL_STRIP_HEAD_LOW_HU = -300.0
CT_SKULL_STRIP_BONE_HU = 150.0
CT_SKULL_STRIP_BONE_DILATE_ITER = 1
CT_SKULL_STRIP_MIN_COMPONENT_VOXELS = 500

# Same embedded split IDs used by recent PHE-SICH baselines when split CSV is absent.
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}

for p in [OUTPUT_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)

# Kaggle/Notebook safe defaults.
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if NNUNET_REPO is not None and str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

print('IS_KAGGLE            :', IS_KAGGLE)
print('KAGGLE_INPUT         :', KAGGLE_INPUT if IS_KAGGLE else 'not Kaggle')
print('WORK_ROOT            :', WORK_ROOT)
print('nnU-Net repo          :', NNUNET_REPO if NNUNET_REPO is not None else 'not found; will use pip-installed nnunetv2')
print('PHE root              :', PHE_ROOT)
print('PHE image dir         :', PHE_IMAGE_DIR)
print('PHE mask dir          :', PHE_MASK_DIR)
print('Split CSV             :', SPLIT_CSV if SPLIT_CSV is not None else 'not found, using embedded val/test IDs')
print('Images                :', len(list(PHE_IMAGE_DIR.glob('*.nii*'))))
print('Masks                 :', len(list(PHE_MASK_DIR.glob('*.nii*'))))
print('nnUNet_raw            :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed   :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results        :', os.environ['nnUNet_results'])
print('nnUNet_n_proc_DA      :', os.environ['nnUNet_n_proc_DA'])
print('nnUNet_compile        :', os.environ['nnUNet_compile'])
print('Dataset               :', DATASET_NAME)
print('CT preprocess profile :', CT_PREPROCESS_PROFILE)
print('CT input windows      :', [(w['name'], w['level'], w['width']) for w in CT_INPUT_WINDOWS])
print('CT skull stripping    :', CT_SKULL_STRIP_ENABLED)


## Optional install

Cell này kiểm tra dependency cần cho nnU-Net. Nếu thiếu `SimpleITK` hoặc dependency chính của nnU-Net, notebook sẽ tự cài bằng `pip` khi `AUTO_INSTALL_MISSING_DEPS = True`.

In [ ]:
INSTALL_NNUNET = False
AUTO_INSTALL_MISSING_DEPS = True

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET:
    import subprocess
    if NNUNET_REPO is None:
        cmd = [sys.executable, '-m', 'pip', 'install', 'nnunetv2']
    else:
        cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# nnU-Net v2 dependencies used by planning/training/I-O. Keep this explicit so local runs fail early.
ensure_import('SimpleITK')
ensure_import('nibabel')
ensure_import('scipy')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')
ensure_import('nnunetv2', 'nnunetv2')

import nnunetv2
print('nnunetv2 import OK:', nnunetv2.__file__)


## Về nhãn ICH/IPH trong bản nnU-Net này

Bản `02_12` hiện **không đưa ICH/IPH vào label**. Lý do là PHE-SICH trong pipeline hiện tại chỉ có manual PHE mask; ICH/IPH của các notebook 3DFF là pseudo-label sinh từ HU/Otsu/ROI quanh PHE.

Với nnU-Net, nếu train multi-class thì label volume phải có integer class rõ ràng, ví dụ:

```text
0 = background
1 = ICH/IPH
2 = PHE
```

Nếu đưa pseudo ICH nhiễu vào ngay, nnU-Net full 3D có thể học rất mạnh theo nhãn sai và làm kết quả PHE khó phân tích. Vì vậy bản baseline đầu tiên nên là PHE-only để trả lời câu hỏi: full 3D nnU-Net trên manual PHE mask đạt được bao nhiêu.

Sau khi có baseline PHE-only, có thể tạo bản sau `02_12b` theo một trong hai hướng:

1. Multi-class pseudo ICH + PHE: dùng lại rule pseudo ICH của `02_10`, tạo label `0/1/2` cho nnU-Net.
2. ICH teacher: train ICH model từ Seg-CQ500/Instance2022, predict ICH trên PHE-SICH, rồi dùng prediction đó làm nhãn phụ sạch hơn HU-rule.

Hướng 2 hợp lý hơn nếu mục tiêu là cải thiện thật sự, vì ICH pseudo hiện tại vẫn là nguồn nhiễu.

## 1. Inspect split và dữ liệu

Nếu `USER_SPLIT_CSV`/auto-detected split CSV có tồn tại, notebook dùng split đó và remap image/mask về PHE-SICH trong Kaggle input. Nếu không có CSV, notebook dùng embedded val/test IDs giống các baseline gần đây: train 99, val 10, test 11.


In [ ]:
def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_value(value):
    text = strip_nii_suffix(value).strip()
    if text.endswith('.0') and text[:-2].isdigit():
        text = text[:-2]
    import re
    m = re.search(r'(\d{4})$', text)
    if not m:
        m = re.search(r'(\d+)$', text)
    if not m:
        raise ValueError(f'Cannot parse scan id from {value}')
    return m.group(1).zfill(4)


def make_nnunet_case_id(value, img_path=None):
    value = '' if value is None or pd.isna(value) else str(value).strip()
    if not value and img_path is not None:
        value = strip_nii_suffix(img_path)
    if value.lower().startswith('phe_'):
        return value
    if value.isdigit():
        return f'phe_{int(value):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in value).strip('_')
    return f'phe_{safe}' if not safe.lower().startswith('phe_') else safe


def split_from_scan_id(scan_id):
    scan_id4 = scan_id_from_value(scan_id)
    if scan_id4 in TEST_SCAN_IDS:
        return 'test'
    if scan_id4 in VAL_SCAN_IDS:
        return 'val'
    return 'train'


def build_nifti_index(folder):
    files = sorted(list(Path(folder).glob('*.nii')) + list(Path(folder).glob('*.nii.gz')))
    out = {}
    for file_path in files:
        try:
            out[scan_id_from_value(file_path)] = file_path
        except ValueError:
            pass
    return out


image_by_scan = build_nifti_index(PHE_IMAGE_DIR)
mask_by_scan = build_nifti_index(PHE_MASK_DIR)
assert image_by_scan, f'No NIfTI images found in {PHE_IMAGE_DIR}'
assert mask_by_scan, f'No NIfTI masks found in {PHE_MASK_DIR}'

rows = []
if SPLIT_CSV is not None and SPLIT_CSV.exists():
    raw_split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
    raw_split_df.columns = [str(c).strip() for c in raw_split_df.columns]
    for raw_row in raw_split_df.to_dict('records'):
        scan_source = raw_row.get('scan_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('patient_id', None)
        if scan_source is None or pd.isna(scan_source) or str(scan_source).strip() == '':
            scan_source = raw_row.get('img_path', raw_row.get('image_path', ''))
        scan_id = scan_id_from_value(scan_source)
        split_value = str(raw_row.get('split', split_from_scan_id(scan_id))).strip().lower()
        if split_value not in {'train', 'val', 'test'}:
            split_value = split_from_scan_id(scan_id)
        img_path = Path(str(raw_row.get('img_path', raw_row.get('image_path', ''))))
        mask_path = Path(str(raw_row.get('phe_mask_path', raw_row.get('label_path', raw_row.get('mask_path', '')))))
        if not img_path.exists():
            img_path = image_by_scan.get(scan_id)
        if not mask_path.exists():
            mask_path = mask_by_scan.get(scan_id)
        rows.append({
            'scan_id': scan_id,
            'case_id': make_nnunet_case_id(scan_id),
            'split': split_value,
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })
else:
    for scan_id, img_path in sorted(image_by_scan.items()):
        mask_path = mask_by_scan.get(scan_id)
        if mask_path is None:
            print('WARNING: missing mask for image scan_id:', scan_id, img_path)
            continue
        rows.append({
            'scan_id': scan_id,
            'case_id': make_nnunet_case_id(scan_id),
            'split': split_from_scan_id(scan_id),
            'img_path': str(img_path),
            'phe_mask_path': str(mask_path),
        })

split_df = pd.DataFrame(rows).drop_duplicates('case_id', keep='first').sort_values(['split', 'scan_id']).reset_index(drop=True)
required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split manifest: {missing_cols}'
assert split_df['case_id'].is_unique, 'case_id must be unique.'
assert split_df['img_path'].map(lambda x: Path(x).exists()).all(), 'Some image paths are missing.'
assert split_df['phe_mask_path'].map(lambda x: Path(x).exists()).all(), 'Some PHE mask paths are missing.'

manifest_csv = OUTPUT_ROOT / '02_12b_phe_sich_split_manifest.csv'
split_df.to_csv(manifest_csv, index=False)

print('Total cases:', len(split_df))
print('Manifest:', manifest_csv)
display(split_df.groupby('split').size().rename('cases').reset_index())
display(split_df.head())


## 2. Convert PHE-SICH sang nnU-Net v2 raw format

nnU-Net cần cấu trúc:

```text
nnUNet_raw/Dataset122_PHESICH_PHE_12b_bwss/
  dataset.json
  imagesTr/phe_0001_0000.nii.gz   # CT brain window, skull-stripped
  imagesTr/phe_0001_0001.nii.gz   # CT subdural window, skull-stripped
  labelsTr/phe_0001.nii.gz
  imagesTs/phe_0003_0000.nii.gz
  imagesTs/phe_0003_0001.nii.gz
  labelsTs/phe_0003.nii.gz
```

Label được ép về binary: `0 background`, `1 PHE`. Không có pseudo-ICH channel trong bản 12b.


In [ ]:
OVERWRITE_RAW_DATASET = True

try:
    import SimpleITK as sitk
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Missing SimpleITK. Rerun the Optional install/dependency cell above, or run: pip install SimpleITK') from exc

try:
    from scipy import ndimage as ndi
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Missing scipy. Rerun the Optional install/dependency cell above, or run: pip install scipy') from exc


def _largest_or_center_component_2d(mask2d, min_voxels=500):
    mask2d = np.asarray(mask2d, dtype=bool)
    labeled, num = ndi.label(mask2d)
    if num == 0:
        return np.zeros(mask2d.shape, dtype=bool)
    yy, xx = np.indices(mask2d.shape)
    center = np.asarray(mask2d.shape, dtype=float) / 2.0
    best_label = None
    best_score = -np.inf
    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        voxels = int(comp.sum())
        if voxels < int(min_voxels):
            continue
        cy = float(yy[comp].mean())
        cx = float(xx[comp].mean())
        dist = np.sqrt((cy - center[0]) ** 2 + (cx - center[1]) ** 2)
        score = voxels / (1.0 + 0.02 * dist)
        if score > best_score:
            best_score = score
            best_label = comp_id
    if best_label is None:
        counts = np.bincount(labeled.ravel())
        counts[0] = 0
        best_label = int(counts.argmax())
    return labeled == best_label


def make_intracranial_mask(ct_arr):
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=-1024.0, posinf=3071.0, neginf=-1024.0)
    if not CT_SKULL_STRIP_ENABLED:
        return np.ones(ct.shape, dtype=bool)
    out = np.zeros(ct.shape, dtype=bool)
    for z in range(ct.shape[0]):
        sl = ct[z]
        head = sl > float(CT_SKULL_STRIP_HEAD_LOW_HU)
        if not head.any():
            continue
        head = ndi.binary_fill_holes(ndi.binary_closing(head, iterations=2))
        head = _largest_or_center_component_2d(head, min_voxels=CT_SKULL_STRIP_MIN_COMPONENT_VOXELS)
        bone = sl > float(CT_SKULL_STRIP_BONE_HU)
        if CT_SKULL_STRIP_BONE_DILATE_ITER > 0:
            bone = ndi.binary_dilation(bone, iterations=int(CT_SKULL_STRIP_BONE_DILATE_ITER))
        brain_candidate = head & ~bone
        brain_candidate = ndi.binary_opening(brain_candidate, iterations=1)
        brain_candidate = _largest_or_center_component_2d(brain_candidate, min_voxels=CT_SKULL_STRIP_MIN_COMPONENT_VOXELS)
        if brain_candidate.any():
            brain_candidate = ndi.binary_fill_holes(brain_candidate)
        out[z] = brain_candidate
    return out.astype(bool)


def window_ct_to_scale(ct_arr, level_hu, width_hu, output_scale=100.0):
    low = float(level_hu) - float(width_hu) / 2.0
    high = float(level_hu) + float(width_hu) / 2.0
    ct = np.nan_to_num(np.asarray(ct_arr, dtype=np.float32), nan=low, posinf=high, neginf=low)
    clipped = np.clip(ct, low, high)
    scaled = (clipped - low) / max(high - low, 1e-6)
    return (scaled * float(output_scale)).astype(np.float32), low, high


def write_float_array_like_reference(arr, reference_img, dst_path: Path):
    out = sitk.GetImageFromArray(np.asarray(arr, dtype=np.float32))
    if out.GetSize() != reference_img.GetSize():
        raise ValueError(f'Size mismatch for {dst_path}: {out.GetSize()} vs ref {reference_img.GetSize()}')
    out.CopyInformation(reference_img)
    sitk.WriteImage(out, str(dst_path))


def write_ct_input_channels(src_path: Path, image_dir: Path, case_id: str):
    ref = sitk.ReadImage(str(src_path))
    ct = sitk.GetArrayFromImage(ref)
    brain_mask = make_intracranial_mask(ct)
    stats = {
        'ct_preprocess_profile': CT_PREPROCESS_PROFILE,
        'ct_input_windows': '|'.join(CT_INPUT_WINDOW_NAMES),
        'brain_mask_voxels': int(brain_mask.sum()),
    }
    channel_record = {}
    for channel_offset, window_spec in enumerate(CT_INPUT_WINDOWS):
        channel_arr, low, high = window_ct_to_scale(
            ct,
            window_spec['level'],
            window_spec['width'],
            output_scale=CT_WINDOW_OUTPUT_SCALE,
        )
        if CT_SKULL_STRIP_ENABLED:
            channel_arr = np.where(brain_mask, channel_arr, 0.0).astype(np.float32)
        dst_path = image_dir / f'{case_id}_{channel_offset:04d}.nii.gz'
        write_float_array_like_reference(channel_arr, ref, dst_path)
        name = str(window_spec['name'])
        channel_record[f'ct_{name}_channel'] = str(dst_path)
        channel_record[f'ct_{name}_window_low_hu'] = float(low)
        channel_record[f'ct_{name}_window_high_hu'] = float(high)
        channel_record[f'ct_{name}_min'] = float(channel_arr.min())
        channel_record[f'ct_{name}_max'] = float(channel_arr.max())
        channel_record[f'ct_{name}_mean'] = float(channel_arr.mean())
    return stats, channel_record


def write_binary_label(src_path: Path, reference_path: Path, dst_path: Path):
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch: reference {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))


if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    seg_src = Path(row.phe_mask_path)
    if row.split == 'test':
        image_dir = imagesTs
        label_dir = labelsTs
    else:
        image_dir = imagesTr
        label_dir = labelsTr

    seg_dst = label_dir / f'{case_id}.nii.gz'
    ct_stats, channel_record = write_ct_input_channels(img_src, image_dir, case_id)
    write_binary_label(seg_src, img_src, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'nnunet_label': str(seg_dst),
        'source_image': str(img_src),
        'source_phe_mask': str(seg_src),
        **ct_stats,
        **channel_record,
    })

channel_names = {}
for channel_offset, window_spec in enumerate(CT_INPUT_WINDOWS):
    name = str(window_spec['name'])
    channel_names[str(channel_offset)] = f"CT_{name}_L{int(window_spec['level'])}_W{int(window_spec['width'])}_{'skullstrip' if CT_SKULL_STRIP_ENABLED else 'noskullstrip'}"

dataset_json = {
    'channel_names': channel_names,
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_manifest = OUTPUT_ROOT / '02_12b_bwss_conversion_manifest.csv'
conversion_df.to_csv(conversion_manifest, index=False)

print('Dataset dir:', DATASET_DIR)
print('CT input windows:', CT_INPUT_WINDOW_NAMES)
print('Total input channels:', len(channel_names), channel_names)
print('imagesTr channel files:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs channel files:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
print('conversion manifest:', conversion_manifest)
display(conversion_df.groupby('split').size().rename('cases').reset_index())
display(pd.DataFrame([dataset_json['channel_names']]).T.rename(columns={0: 'channel_name'}))


## 3. Helper gọi entrypoint của nnU-Net

Các command `nnUNetv2_*` thường chạy từ terminal. Trong notebook này gọi trực tiếp entrypoint Python để không phụ thuộc PATH của Windows.

In [ ]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 4. Plan and preprocess

`RUN_PLAN_PREPROCESS` đang bật sẵn để chạy thật. Bản này mặc định preprocess `3d_fullres`. Nếu GPU/RAM yếu, đổi `CONFIGURATION = '2d'`.

In [ ]:
CONFIGURATION = '3d_fullres'  # alternatives: '2d', '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    # Planner calls torch.set_num_threads(get_allowed_n_proc_DA()), so this must be >= 1.
    # Training cell below will switch it back to 0 to avoid Windows/Jupyter background-worker crashes.
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run nnU-Net planning/preprocessing.')


## 5. Ghi manual split cho fold 0

Sau khi plan/preprocess, chạy cell này để ép fold 0 dùng đúng split mới:

- train: 99 case
- val: 10 case
- test: 11 case, nằm ngoài training và chỉ dùng predict/evaluate

In [ ]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


## 6. Train nnU-Net fold 0

Các lựa chọn trainer:

- `nnUNetTrainer_5epochs`: smoke test nhanh, chỉ để kiểm tra pipeline.
- `nnUNetTrainer_250epochs`: bản nhẹ hơn default, hợp lý để thử trước.
- `nnUNetTrainer`: default nnU-Net 1000 epochs, nặng nhất nhưng đúng baseline chuẩn hơn.

`RUN_TRAIN` đang bật sẵn. Mặc định dùng `nnUNetTrainer_250epochs`, giống bản 12 để giữ công bằng. Cell này tự chuyển sang fresh training nếu chưa có checkpoint.


In [ ]:
RUN_TRAIN = True
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

EFFECTIVE_CONTINUE_TRAINING = CONTINUE_TRAINING
if CONTINUE_TRAINING:
    available_checkpoints = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth'))
    if not available_checkpoints:
        print('CONTINUE_TRAINING=True but no checkpoint exists yet; starting a fresh training run.')
        EFFECTIVE_CONTINUE_TRAINING = False
    else:
        print('Continuing from available checkpoints:', available_checkpoints)

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('Model dir:', MODEL_DIR)
    print('Fold dir :', FOLD_DIR)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    print('continue_training:', EFFECTIVE_CONTINUE_TRAINING)
    # Windows/Jupyter guard: ensure the trainer output parent exists before nnU-Net copies fingerprint files.
    import shutil as _shutil
    if not hasattr(_shutil, '_codex_original_copyfile'):
        _shutil._codex_original_copyfile = _shutil.copyfile

    def _copyfile_with_parent(src, dst, *args, **kwargs):
        if str(src).endswith('dataset_fingerprint.json'):
            Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return _shutil._codex_original_copyfile(src, dst, *args, **kwargs)

    _shutil.copyfile = _copyfile_with_parent
    try:
        run_training(
            str(DATASET_ID),
            CONFIGURATION,
            str(FOLD),
            trainer_class_name=TRAINER,
            plans_identifier=PLANS,
            export_validation_probabilities=EXPORT_VAL_NPZ,
            continue_training=EFFECTIVE_CONTINUE_TRAINING,
            device=device,
        )
    finally:
        _shutil.copyfile = _shutil._codex_original_copyfile
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


## 7. Predict test set và evaluate

Cell này predict 11 test case trong `imagesTs` và so với `labelsTs`. Nếu dùng trainer/config khác lúc train, phải giữ cùng `TRAINER`, `CONFIGURATION`, `PLANS` ở đây.

In [ ]:
RUN_PREDICT = True
RUN_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'  # auto: best -> final -> latest; or set checkpoint_best.pth/checkpoint_final.pth manually

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
print('Model dir          :', MODEL_DIR)
print('Fold dir           :', FOLD_DIR)
print('Using checkpoint   :', RESOLVED_CHECKPOINT)

PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

if RUN_PREDICT:
    import gc
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

    # Do not call nnUNetv2_predict entrypoint inside a post-training notebook kernel.
    # The entrypoint calls torch.set_num_interop_threads(1), which fails after PyTorch parallel work has started.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Predict device:', device)

    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PREDICT=True to run inference.')

if RUN_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


## 8. Đọc kết quả summary

nnU-Net summary mặc định có Dice và IoU. Các metric như HD95/ASSD/NSD/RVD có thể bổ sung sau bằng evaluator đang dùng ở các notebook 3DFF.

In [ ]:
if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print('Foreground mean:')
    print(json.dumps(summary.get('foreground_mean', {}), indent=2))
    print('\nPHE label metrics:')
    print(json.dumps(summary.get('mean', {}).get('1', {}), indent=2))
else:
    print('No summary yet:', SUMMARY_JSON)
    print('Run train -> predict -> evaluate first.')


## Ghi chú so sánh với 02_12, 02_12b local và 19b

Bản `02_12b_kaggle` là PHE-only nnU-Net baseline giống `02_12b` local, nhưng chạy trên Kaggle:

1. Brain window L=40/W=120.
2. Subdural window L=75/W=215.
3. Skull stripping nhẹ bằng mask nội sọ tạo từ CT.
4. Không dùng pseudo ICH, không dùng ICH teacher, không dùng PHE mask để tạo input.

Khi báo cáo, so sánh:

- `02_12`: CT raw 1 channel, PHE-only.
- `02_12b`: CT multi-window/skull-strip 2 channels, PHE-only.
- `02_19b`: cùng tiền xử lý CT nhưng có thêm pseudo-ICH prior CT-only.

Nếu 12b tăng rõ so với 12, tiền xử lý CT là yếu tố chính. Nếu 19b còn tăng thêm so với 12b, pseudo-ICH prior mới có đóng góp riêng.
